In [1]:
import sys
from pathlib import Path

#Ensure project root is available for imports
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

In [2]:
#Imports
import numpy as np

#Import functions from .py files
from src.data_loader import load_data
from src.feature_engineering import add_engineered_features
from src.inference import predict_from_xml 

from sklearn.metrics import mean_absolute_error, mean_squared_error

#load data
#Database path
db_path = r"C:\Users\jacov\anaconda_projects\Marvel Insurance\data\superhero_events_db.duckdb"

df = load_data(db_path)
df = add_engineered_features(df)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (91, 11)


,index,name,CreditInfo,annual_public_destruction_events,credit_score,age,num_superpowers,num_properties,num_credit_cards,total_credit_limit,xml_parse_error
0,0,Nyx,"<?xml version=""1.0"" encoding=""UTF-8"" ?><superh...",23,436.0,47.0,3,6,1,1117,0
1,1,Nova,"<?xml version=""1.0"" encoding=""UTF-8"" ?><superh...",24,215.0,29.0,3,2,4,154863,0
2,2,Psylocke,"<?xml version=""1.0"" encoding=""UTF-8"" ?><superh...",18,344.0,62.0,2,1,4,109612,0
3,3,Cerberus,"<?xml version=""1.0"" encoding=""UTF-8"" ?><superh...",8,460.0,76.0,2,0,0,0,0
4,4,Iron Man,"<?xml version=""1.0"" encoding=""UTF-8"" ?><superh...",12,475.0,77.0,3,0,2,166017,0


In [3]:
#List of all new applicatant - predicted

#Superheroes without known destruction events
to_predict = df[df["annual_public_destruction_events"].isnull()]

print("Number of new applicants:", len(to_predict))
print("\nSuperheroes to predict for:")
print(to_predict["name"].tolist())

Number of new applicants: 21

Superheroes to predict for:
['Elektra', 'Juggernaut', 'Ant-Man', 'Thanos', 'Bucky Barnes', 'X-Men', 'Xena', 'Mimir', 'Rocket Raccoon', 'Jormungandr', 'Scarlet Witch', 'Zephyrus', 'Lorelei', 'Zeus', 'Nick Fury', 'Balder', 'Ororo Munroe', 'Idunn', 'Baldur', 'Thor', 'Daredevil']


In [4]:
#Test 3 non-client superheroes
heroes_to_test = ["Iron Man", "Ant-Man", "Scarlet Witch"]

for hero in heroes_to_test:
    try: 
        sample_xml = df.loc[df["name"] == hero, "CreditInfo"].iloc[0]
        prediction = predict_from_xml(sample_xml) #Note that the prediction function round up.
        print(f"{hero}: Predicted destruction events = {prediction: .2f}")
    except Exception as e:
        print(f"{hero}: Error -> {e}")

Iron Man: Predicted destruction events =  12.00
Ant-Man: Predicted destruction events =  29.00
Scarlet Witch: Predicted destruction events =  20.00


In [18]:
#Predictions for new applicants

#Remove malformed XML if present
if "xml_parse_error" in to_predict.columns: to_predict = to_predict[to_predict["xml_parse_error"] == 0]

#Generate predictions
to_predict["predicted_destruction"] = to_predict["CreditInfo"].apply(lambda xml: predict_from_xml(xml))

#Clean table
underwriting_table = to_predict[["name", "predicted_destruction", "credit_score", "age", "num_properties"]].sort_values("name", ascending=True)

print("Prediction on new clients")
underwriting_table.head(22)

Prediction on new clients


,name,predicted_destruction,credit_score,age,num_properties
72,Ant-Man,29,268.0,21.0,4
85,Balder,23,469.0,54.0,8
88,Baldur,27,230.0,25.0,12
74,Bucky Barnes,20,499.0,68.0,12
90,Daredevil,26,339.0,26.0,5
70,Elektra,18,325.0,63.0,0
87,Idunn,20,658.0,65.0,10
79,Jormungandr,17,548.0,36.0,2
71,Juggernaut,19,690.0,53.0,7
82,Lorelei,15,396.0,79.0,3


In [16]:
#Test on our client superheroes to evaluate model performance

#Keep only heroes with known destruction events - clients
df_existing = df[df["annual_public_destruction_events"].notnull()].copy()

#Remove possible malformed XML rows 
if "xml_parse_error" in df_existing.columns: df_existing = df_existing[df_existing["xml_parse_error"] == 0]

#Generate predictions
df_existing["predicted_destruction"] = df_existing["CreditInfo"].apply(lambda xml: predict_from_xml(xml))

#Calculate errors
df_existing["error"] = (df_existing["predicted_destruction"] - df_existing["annual_public_destruction_events"])
df_existing["abs_error"] = df_existing["error"].abs()

mae = mean_absolute_error(df_existing["annual_public_destruction_events"], df_existing["predicted_destruction"])
rmse = np.sqrt(mean_squared_error(df_existing["annual_public_destruction_events"], df_existing["predicted_destruction"]))

#Table
comparison_table = df_existing[[
    "name",
    "annual_public_destruction_events",
    "predicted_destruction",
    "credit_score",
    "age",
    "num_properties"
]].sort_values("name", ascending=True)

print("Model Evaluation on Existing Clients")
print(" ")
print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")

comparison_table.tail(25)


Model Evaluation on Existing Clients
 
MAE:  2.058
RMSE: 2.823


,name,annual_public_destruction_events,predicted_destruction,credit_score,age,num_properties
68,Phoenix,29,28,734.0,55.0,13
40,Poseidon,15,17,404.0,42.0,2
5,Professor X,21,20,567.0,70.0,11
2,Psylocke,18,18,344.0,62.0,1
36,Quicksilver,13,15,640.0,57.0,0
55,Ragnarok,30,30,36.0,63.0,9
25,Red Skull,21,21,633.0,70.0,10
56,Rogue,11,15,727.0,81.0,1
27,Sif,10,17,684.0,42.0,9
19,Skurge,18,19,691.0,81.0,12
